Download de alguns datasets em formato grib. Cada arquivo representa um instante de tempo com data e hora fixa, contendo um conjunto de datasets com diversas variáveis meterológicas. 

In [1]:
#pip install netcdf4

In [2]:
import os
import requests
import numpy as np
import xarray as xr
import cfgrib
from scipy.ndimage import gaussian_filter
from tqdm import tqdm

In [3]:
# ==============================
# CONFIGURAÇÃO
# ==============================

DATE = "20240101"
HOURS = range(0,6)

VARIABLES = [
    "t2m",
    "u10",
    "v10",
    "sp",
]

PATCH_SIZE = 128
STRIDE = 128

DATA_DIR = "hrrr_data"
os.makedirs(DATA_DIR,exist_ok=True)

In [4]:
# ==============================
# 1 DOWNLOAD HRRR
# ==============================

def download_hrrr(date,hour):

    url=f"https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr.{date}/conus/hrrr.t{hour:02d}z.wrfsfcf00.grib2"

    file=f"{DATA_DIR}/hrrr_{date}_{hour}.grib2"

    if os.path.exists(file):
        return file

    r=requests.get(url)

    with open(file,"wb") as f:
        f.write(r.content)

    return file

In [5]:
# ==============================
# 2 DETECTAR VARIÁVEIS
# ==============================

def load_variables(file):

    datasets = cfgrib.open_datasets(
        file,
        backend_kwargs={"indexpath":""}
    )

    variables={}

    for ds in datasets:
        for v in ds.data_vars:
            variables[v]=ds[v]

    return variables


In [6]:
# ==============================
# 3 EXTRAIR CAMPOS
# ==============================

def extract_fields(files):

    stacks={v:[] for v in VARIABLES}

    for file in tqdm(files):

        vars=load_variables(file)

        for v in VARIABLES:

            if v in vars:

                field=vars[v].values

                stacks[v].append(field)

    for v in stacks:
        stacks[v]=np.stack(stacks[v])

    return stacks

In [7]:
# ==============================
# 4 EXTRAIR PATCHES
# ==============================

def extract_patches(field,size,stride):

    patches=[]

    T,H,W=field.shape

    for t in range(T):

        f=field[t]

        for i in range(0,H-size,stride):
            for j in range(0,W-size,stride):

                patches.append(f[i:i+size,j:j+size])

    return np.array(patches)

In [8]:
# ==============================
# 5 GERAR DATASET
# ==============================

def build_dataset(fields):

    inputs={}
    outputs={}

    for v,data in fields.items():

        patches=extract_patches(data,PATCH_SIZE,STRIDE)

        hr=patches
        lr=gaussian_filter(hr,sigma=(0,4,4))

        inputs[v]=lr
        outputs[v]=hr

    return inputs,outputs



In [9]:
# ==============================
# 6 SALVAR NETCDF
# ==============================

def save_dataset(inputs,outputs):

    samples=list(inputs.values())[0].shape[0]

    coords={
        "sample":np.arange(samples),
        "y":np.arange(PATCH_SIZE),
        "x":np.arange(PATCH_SIZE)
    }

    ds_input=xr.Dataset(
        {v:(["sample","y","x"],inputs[v]) for v in inputs},
        coords=coords
    )

    ds_output=xr.Dataset(
        {v:(["sample","y","x"],outputs[v]) for v in outputs},
        coords=coords
    )

    elevation=np.random.randn(PATCH_SIZE,PATCH_SIZE)
    landmask=(np.random.rand(PATCH_SIZE,PATCH_SIZE)>0.5).astype(float)

    ds_inv=xr.Dataset(
        {
            "elevation":(["y","x"],elevation),
            "landmask":(["y","x"],landmask)
        }
    )

    ds_input.to_netcdf("corrdiff_dataset.nc",group="input")
    ds_output.to_netcdf("corrdiff_dataset.nc",group="output",mode="a")
    ds_inv.to_netcdf("corrdiff_dataset.nc",group="invariant",mode="a")


In [10]:
# ==============================
# PIPELINE
# ==============================

def main():

    files=[]

    for h in HOURS:
        f=download_hrrr(DATE,h)
        files.append(f)

    fields=extract_fields(files)

    inputs,outputs=build_dataset(fields)

    save_dataset(inputs,outputs)

    print("Dataset gerado: corrdiff_dataset.nc")


if __name__=="__main__":
    main()

  0%|          | 0/6 [00:00<?, ?it/s]

/home/sangonvi/Cefet/repositories/rionowcastdf/.venv/lib/python3.12/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/home/sangonvi/Cefet/repositories/rionowcastdf/.venv/lib/python3.12/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicit

Dataset gerado: corrdiff_dataset.nc


In [11]:
import xarray as xr
import json
import numpy as np

ds = xr.open_dataset("corrdiff_dataset.nc", group="input")

stats = {}

stats["input"] = {}
for v in ds.data_vars:
    data = ds[v].values
    stats["input"][v] = {
        "mean": float(np.mean(data)),
        "std": float(np.std(data))
    }

stats["output"] = stats["input"]
stats["invariant"] = {
    "elev_mean":{"mean":0,"std":1},
    "lsm_mean":{"mean":0,"std":1}
}

json.dump(stats,open("stats.json","w"),indent=2)